In [ ]:
!pip install -q tensorflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 644.9/644.9 MB 729.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 94.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 133.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 130.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.3/106.3 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 121.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.5/224.5 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.3 MB/s eta 0:00:00


In [ ]:
import tensorflow as tf
import numpy as np
import os
import math
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.data import TFRecordDataset
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Flatten, Reshape, Conv2D, Conv2DTranspose
from tensorflow.keras.optimizers import Adam
from google.colab import drive

In [ ]:
try:
    tpu = tf.distribute.cluster_resolver.TPUClusterResolver()
    print("TPU detected:", tpu.cluster_spec().as_dict())
except ValueError:
    print("No TPU detected.")

No TPU detected.


In [ ]:
tf.config.list_physical_devices()

[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]

In [ ]:
try:
    resolver = tf.distribute.cluster_resolver.TPUClusterResolver()
    print("TPU detected:", resolver.master())
except ValueError:
    print("No TPU detected.")

No TPU detected.


In [ ]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"rumiyyaalili","key":"5a48756827bd010b7b5a1f5561da34e8"}'}

In [ ]:
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [12]:
!kaggle datasets download -d rumiyyaalili/brst-dataset-binary

Dataset URL: https://www.kaggle.com/datasets/rumiyyaalili/brst-dataset-binary
License(s): unknown
 99% 984M/993M [00:11<00:00, 102MB/s]
100% 993M/993M [00:11<00:00, 93.9MB/s]


In [13]:
!unzip -qq brst-dataset-binary

In [14]:
# Create directories
os.makedirs("images", exist_ok=True)

In [ ]:
# import tensorflow as tf
# import os

# # Path to your images folder
# image_folder = "/content/InBreast_Aligned_Images"
# tfrecord_file = "/content/brst_dataset.binary"

# # Function to serialize images into TFRecord
# def serialize_example(image_path):
#     with open(image_path, "rb") as img_file:
#         image_bytes = img_file.read()

#     feature = {
#         "image_raw": tf.train.Feature(bytes_list=tf.train.BytesList(value=[image_bytes]))
#     }
#     example_proto = tf.train.Example(features=tf.train.Features(feature=feature))
#     return example_proto.SerializeToString()

# # Write TFRecord file
# with tf.io.TFRecordWriter(tfrecord_file) as writer:
#     for filename in os.listdir(image_folder):
#         if filename.endswith((".png", ".jpg", ".jpeg")):
#             image_path = os.path.join(image_folder, filename)
#             tfrecord_entry = serialize_example(image_path)
#             writer.write(tfrecord_entry)
# print("TFRecord file created successfully!")

In [15]:
# Inspect the first few records to check the feature names in the TFRecord file
raw_dataset = tf.data.TFRecordDataset("brst_dataset.tfrecord")

# Try to print out the first few raw records to understand their structure
for raw_example in raw_dataset.take(5):
    print(raw_example)  # Prints raw binary data of the TFRecord example

tf.Tensor(b'\n\xc8x\n\xc5x\n\timage_raw\x12\xb7x\n\xb4x\n\xb1x\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x01\x00\x00\x00\x01\x00\x08\x00\x00\x00\x00y\x19\xf7\xba\x00\x00;\xf8IDATx\x9c\xe5\xbdY\xcc\xaeYv\xdf\xb5\xa6\xbd\x9f\xe1\x1d\xbf\xe1\xcc\xe7\xd4\xa9\xb9k\xean\xb7\xdbm;\xddI:\xc1\x19L\x90b\x04Q\x82\x84\xa2\x90+$\xa4HD\xe4\n\t\x85\\\x00\x12BB(\x97\xdc\x10\x85I\x16\x84\x10)$1N\x10\x02\xe3\xc4\tq\xdb\xddn\xbbz\xa8\xaa\xaeSu\xc6o|\xa7\xe7\xd9\xc3Z\x8b\x8b\xea\x18\x93\x10\x13w\xe8\xae\xf7\xeb\xfa_\xbd\xaf>\xe9\xd1\xfb\xfc\xbe\xb5\xf7Z{\xed\xbd\xd6\x06\xf8\x84\x0b?\xee\x1f\xf0;\xd0-\x19\x9aJN\n\x90\xe0\x9aL\xacYGL\xa5c\xf0\xb0\x81~GR`RiR^\xb4\xa7\xe1N\x9b`i\xb7\xd7w\xbeq\x19\xa7\xb3\xd2>j\xa6\xf4\xfc%\x1cl|\xfa\xf8\xd5\x0f\xaf=\x9a\xbe\xf6\xf5\x1b\xd7\xba\xc9\xe2*\x01\xb8\xd6\x14R\x82P\xf4\xf0Y\xed\'M\xd0.\xd5\x1a\xac\xf8,\x8c\xe3\xa44\xd9Y\x89\xa4\xd1Y?\xd0+72\x9fM\x07\xaf\xddz\xb1\xed\xe8\xe8$\xde~\xfa\xdcE\x83\x07\xbb\xa6\xf6\x03\x9e\xdf\xbb\xb9\xbb\xd5Rw\x95\x00\xf4\xd7a\x87\xe0\x0e

In [16]:
raw_dataset = tf.data.TFRecordDataset(["brst_dataset.tfrecord"])
for raw_record in raw_dataset.take(1):  # Take a few records to inspect
    example = tf.train.Example()
    example.ParseFromString(raw_record.numpy())
    print(example)

features {
  feature {
    key: "image_raw"
    value {
      bytes_list {
        value: "\211PNG\r\n\032\n\000\000\000\rIHDR\000\000\001\000\000\000\001\000\010\000\000\000\000y\031\367\272\000\000;\370IDATx\234\345\275Y\314\256Yv\337\265\246\275\237\341\035\277\341\314\347\324\251\271k\352n\267\333m;\335I:\301\031L\220b\004Q\202\204\242\220+$\244HD\344\n\t\205\\000\022BB(\227\334\020\205I\026\204\020)$1N\020\002\343\304\tq\333\335n\273z\250\252\256Su\306o|\247\347\331\303Z\213\213\352\030\223\020\023w\350\256\367\353\372_\275\257>\351\321\373\374\276\265\367Z{\355\275\326\006\370\204\013?\356\037\360;\320-\031\232JN\n\220\340\232L\254YGL\245c\360\260\201~GR`RiR^\264\247\341N\233`i\267\327w\276q\031\247\263\322>j\246\364\374%\034l|\372\370\325\017\257=\232\276\366\365\033\327\272\311\342*\001\270\326\024R\202P\364\360Y\355'M\320.\325\032\254\370,\214\343\2444\331Y\211\244\321Y?\320+72\237M\007\257\335z\261\355\350\350$\336~\372\334E\203\007\273\246\366\003\236\337\273\271\273\325Rw\2

In [17]:
def _parse_function(proto):
    # Define the feature description for 'image_raw' which is stored as bytes
    keys_to_features = {'image_raw': tf.io.FixedLenFeature([], tf.string)}  # Use 'image_raw' key instead of 'image'

    # Parse the TFRecord
    parsed_features = tf.io.parse_single_example(proto, keys_to_features)

    # Decode the PNG image from the byte string
    image = tf.io.decode_png(parsed_features['image_raw'], channels=1)  # Decode the PNG and ensure it's single-channel (grayscale)

    # Normalize the image to the range [-1, 1] for GAN training
    image = tf.cast(image, tf.float32) / 127.5 - 1.0  # Normalize to [-1, 1]

    return image

In [18]:
# Set parameters
n_epochs = 200  # Number of epochs of training
batch_size = 64  # Size of the batches
lr_g = 0.0002  # Adam: learning rate for generator
lr_d = 0.00005 # learning rate for discriminator was 0.001
b1 = 0.9  # Adam: decay of first-order momentum of gradient
b2 = 0.999  # Adam: decay of second-order momentum of gradient
latent_dim = 200  # Dimensionality of the latent space was 256
img_size = 256  # Size of each image dimension
channels = 1  # Number of image channels (grayscale)
n_critic = 5  # Number of training steps for discriminator per iteration reduced to 5
#clip_value = 0.01  # Lower and upper clip values for discriminator weights
sample_interval = 400  # Interval between image samples
lambda_gp = 10.0 # Define the gradient penalty weight

In [19]:
import tensorflow as tf
from tensorflow.keras import layers

class Generator(tf.keras.Model):
    def __init__(self, latent_dim, img_shape):
        super(Generator, self).__init__()
        self.img_shape = img_shape  # (height, width, channels)
        self.channels = img_shape[-1]  # Extract channels (TensorFlow uses NHWC format)

        def block(units, normalize=True):
            block_layers = [layers.Dense(units)]
            if normalize:
                block_layers.append(layers.BatchNormalization(momentum=0.8))
            block_layers.append(layers.LeakyReLU(alpha=0.2))
            return block_layers

        # Fully connected layers
        self.model_fc = tf.keras.Sequential(
            block(128, normalize=False) +
            block(256) +
            block(512) +
            block(1024) +
            [layers.Dense(tf.math.reduce_prod(img_shape), activation="tanh")]
        )

        # Convolutional refinement part
        self.conv_refine = tf.keras.Sequential([
            layers.Conv2D(32, kernel_size=3, strides=1, padding="same"),
            layers.LeakyReLU(alpha=0.2),
            layers.Conv2D(32, kernel_size=3, strides=1, padding="same"),
            layers.LeakyReLU(alpha=0.2),
            layers.Conv2D(self.channels, kernel_size=3, strides=1, padding="same", activation="tanh"),
        ])

    def call(self, z):
        # Generate image from latent vector
        img = self.model_fc(z)

        # Reshape into (batch_size, height, width, channels)
        img = tf.reshape(img, (-1, self.img_shape[0], self.img_shape[1], self.channels))

        # Apply CNN refinement
        img = self.conv_refine(img)

        return img

In [20]:
import tensorflow as tf
from tensorflow.keras import layers

class Discriminator(tf.keras.Model):
    def __init__(self, img_shape):
        super(Discriminator, self).__init__()
        self.img_shape = img_shape  # (height, width, channels)

        self.conv_layers = tf.keras.Sequential([
            layers.Conv2D(64, kernel_size=4, strides=2, padding="same"),
            layers.LeakyReLU(alpha=0.2),

            layers.Conv2D(128, kernel_size=4, strides=2, padding="same"),
            layers.LeakyReLU(alpha=0.2),

            layers.Conv2D(256, kernel_size=4, strides=2, padding="same"),
            layers.LeakyReLU(alpha=0.2),

            layers.Conv2D(512, kernel_size=4, strides=2, padding="same"),
            layers.LeakyReLU(alpha=0.2),
        ])

        self.global_pool = layers.GlobalAveragePooling2D()  # Compute flattened size dynamically

        self.fc_layers = tf.keras.Sequential([
            layers.Dense(512),
            layers.LeakyReLU(alpha=0.2),
            layers.Dense(1)  # Output a single scalar (real/fake)
        ])

    def call(self, img):
        features = self.conv_layers(img)  # Apply convolutional layers
        features_flat = self.global_pool(features)  # Flatten automatically
        validity = self.fc_layers(features_flat)  # Fully connected layers
        return validity

In [21]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

def generate_and_display_images(generator, num_images=25, latent_dim=100, nrow=5):
    """
    Generate and display images using the trained generator (TensorFlow version).

    Args:
        generator (tf.keras.Model): The trained generator model.
        num_images (int): Number of images to generate.
        latent_dim (int): Dimensionality of the latent space.
        nrow (int): Number of images per row in the grid.
    """
    # Generate latent vectors
    z = tf.random.normal([num_images, latent_dim])  # Shape: (num_images, latent_dim)

    # Generate images
    generated_images = generator(z, training=False)  # Shape: (num_images, height, width, channels)

    # Convert tensor to NumPy for visualization
    generated_images = generated_images.numpy()

    # Normalize to [0, 1] for display
    generated_images = (generated_images + 1) / 2.0

    # Determine grid size
    fig, axes = plt.subplots(nrow, nrow, figsize=(10, 10))
    for i, ax in enumerate(axes.flat):
        if i < num_images:
            ax.imshow(generated_images[i, :, :, 0], cmap="gray")  # Grayscale image
            ax.axis("off")
    plt.show()

In [22]:
import tensorflow as tf
import os
from tensorflow.keras.optimizers import Adam
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# TPU Initialization
try:
    tpu = tf.distribute.cluster_resolver.TPUClusterResolver()
    tf.config.experimental_connect_to_cluster(tpu)
    tf.tpu.experimental.initialize_tpu_system(tpu)
    strategy = tf.distribute.TPUStrategy(tpu)
    print("Running on TPU:", tpu.master())
except ValueError:
    strategy = tf.distribute.get_strategy()
    print("Running on CPU/GPU")

# Define directories inside Google Drive
base_image_dir = "/content/drive/My Drive/TF_GAN_Generated_Images"
save_dir = "/content/drive/My Drive/TF_GAN_Weights"
os.makedirs(base_image_dir, exist_ok=True)
os.makedirs(save_dir, exist_ok=True)

# Training hyperparameters
checkpoint_interval = 10  # Save every 10 epochs
lr_g = 0.0002
lr_d = 0.0002
b1, b2 = 0.5, 0.999
latent_dim = 100
img_shape = (256, 256, 1)  # Make sure the shape matches the TFRecord images
lambda_gp = 10.0
batch_size = 64
epochs = 100
n_critic = 5  # Train discriminator 5x per generator step

# TFRecord Parsing Function
def _parse_function(proto):
    keys_to_features = {'image': tf.io.FixedLenFeature([], tf.string)}
    parsed_features = tf.io.parse_single_example(proto, keys_to_features)
    image = tf.io.decode_png(parsed_features['image'], channels=1)  # Ensure channels=1 for grayscale
    image = tf.cast(image, tf.float32) / 127.5 - 1.0  # Normalize to [-1, 1]
    image = tf.image.resize(image, img_shape[:2])  # Resize to the expected shape
    return image

# Load TFRecord Dataset
def load_tfrecord_dataset(tfrecord_file, batch_size):
    dataset = tf.data.TFRecordDataset([tfrecord_file])
    dataset = dataset.map(_parse_function, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.shuffle(10000).batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return dataset

# Instantiate the models
generator = Generator(latent_dim, img_shape)
discriminator = Discriminator(img_shape)

# Optimizers
optimizer_G = Adam(learning_rate=lr_g, beta_1=b1, beta_2=b2)
optimizer_D = Adam(learning_rate=lr_d, beta_1=b1, beta_2=b2)

# Checkpointing
checkpoint = tf.train.Checkpoint(generator=generator, discriminator=discriminator,
                                 optimizer_G=optimizer_G, optimizer_D=optimizer_D)
checkpoint_manager = tf.train.CheckpointManager(checkpoint, directory=save_dir, max_to_keep=5)

# Restore checkpoint if available
if checkpoint_manager.latest_checkpoint:
    checkpoint.restore(checkpoint_manager.latest_checkpoint)
    print("Restored from latest checkpoint.")

# Gradient Penalty
def compute_gradient_penalty(D, real_images, fake_images):
    alpha = tf.random.uniform([real_images.shape[0], 1, 1, 1], 0.0, 1.0)
    interpolates = alpha * real_images + (1 - alpha) * fake_images
    with tf.GradientTape() as tape:
        tape.watch(interpolates)
        d_interpolates = D(interpolates, training=True)
    gradients = tape.gradient(d_interpolates, interpolates)
    gradients = tf.reshape(gradients, [gradients.shape[0], -1])
    gradient_penalty = tf.reduce_mean(tf.square(tf.norm(gradients, axis=1) - 1.0))
    return gradient_penalty

# Loss Functions
def discriminator_loss(real_validity, fake_validity, gradient_penalty):
    return -tf.reduce_mean(real_validity) + tf.reduce_mean(fake_validity) + lambda_gp * gradient_penalty

def generator_loss(fake_validity):
    return -tf.reduce_mean(fake_validity)

# Training Function
def train(dataset, generator, discriminator, epochs, checkpoint_interval):
    for epoch in range(epochs):
        for i, batch in enumerate(dataset):
            real_images = batch

            # Train Discriminator
            with tf.GradientTape() as tape_D:
                z = tf.random.normal([real_images.shape[0], latent_dim])
                fake_images = generator(z, training=True)
                real_validity = discriminator(real_images, training=True)
                fake_validity = discriminator(fake_images, training=True)
                gradient_penalty = compute_gradient_penalty(discriminator, real_images, fake_images)
                d_loss = discriminator_loss(real_validity, fake_validity, gradient_penalty)
            gradients_D = tape_D.gradient(d_loss, discriminator.trainable_variables)
            optimizer_D.apply_gradients(zip(gradients_D, discriminator.trainable_variables))

            # Train Generator every n_critic steps
            if i % n_critic == 0:
                with tf.GradientTape() as tape_G:
                    z = tf.random.normal([real_images.shape[0], latent_dim])
                    fake_images = generator(z, training=True)
                    fake_validity = discriminator(fake_images, training=True)
                    g_loss = generator_loss(fake_validity)
                gradients_G = tape_G.gradient(g_loss, generator.trainable_variables)
                optimizer_G.apply_gradients(zip(gradients_G, generator.trainable_variables))

        print(f"[Epoch {epoch}/{epochs}] [D loss: {d_loss}] [G loss: {g_loss}]")

        # Save images & checkpoints
        if epoch % checkpoint_interval == 0:
            save_generated_images(generator, epoch)
            checkpoint_manager.save()
            print(f"Checkpoint saved at epoch {epoch}.")

# Save Generated Images
def save_generated_images(generator, epoch, num_images=25):
    z = tf.random.normal([num_images, latent_dim])
    generated_images = generator(z, training=False)
    for i in range(num_images):
        img_path = os.path.join(base_image_dir, f"epoch_{epoch}_img_{i}.png")
        tf.keras.preprocessing.image.save_img(img_path, (generated_images[i].numpy() + 1) * 127.5)  # Rescale
    print(f"Saved generated images for epoch {epoch}.")

# Load TFRecord Dataset
tfrecord_file = "/content/brst_dataset.tfrecord"
dataset = load_tfrecord_dataset(tfrecord_file, batch_size=batch_size)

# Start Training
with strategy.scope():  # Runs training on TPU
    train(dataset, generator, discriminator, epochs, checkpoint_interval)

Mounted at /content/drive
Running on CPU/GPU


/usr/local/lib/python3.11/dist-packages/keras/src/layers/activations/leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


InvalidArgumentError: {{function_node __wrapped__IteratorGetNext_output_types_1_device_/job:localhost/replica:0/task:0/device:CPU:0}} Error in user-defined function passed to ParallelMapDatasetV2:12 transformation with iterator: Iterator::Root::Prefetch::BatchV2::Shuffle::ParallelMapV2: Feature: image (data type: string) is required but could not be found.
	 [[{{node ParseSingleExample/ParseExample/ParseExampleV2}}]] [Op:IteratorGetNext] name: 